# Praxis · Reproduce our validation — *Diversified reward-risk parity* (DJIA, #359491)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/359491-reward-risk-parity.ipynb)

**Paper:** *Diversified reward-risk parity in portfolio construction* (2021), by Jaehyung Choi, Hyangju Kim & Young Shin Kim — [arXiv 2106.09055](https://arxiv.org/abs/2106.09055).

This notebook **reproduces our exact verdict** on the paper's headline claim, from **public data only** — Yahoo Finance, which is the paper's *own disclosed source*. It:
1. rebuilds the paper's **point-in-time Dow universe** (their Table 1, verbatim — including the holes we document: GM and Kraft were never servable from Yahoo, and Chevron is missing from the table entirely),
2. runs the **Calmar-parity rule** (score each member by trailing-6-month return ÷ drawdown, weight proportionally, hold 6 months in overlapping sleeves) against the **equal-weight benchmark**,
3. prints **claimed → measured** — and the number the paper never reports: the **t-statistic of the claimed edge**.

**Our published verdict: OVERSTATED — the mechanics replicate (10.98% vs claimed 10.32%), but the +48bp edge over equal weight is statistical noise (t = 0.69 over 22 years), and what tilt exists is a momentum loading (β 0.18, t = 4.0; residual α t = 0.4).** Full report:
https://market-graphs.com/praxis/papers/359491

> **Same public data, same logic as our engine — this lands on our exact published numbers.** It runs the headline reproduction; the full **C0–C7 gate suite** (46-config deflation, PBO, crisis windows, factor spanning) lives in the report. Runs in ~1 min.


In [ ]:
!pip install -q "yfinance==0.2.65"


In [ ]:
"""Praxis · reproduce our validation of paper 359491 "Diversified reward-risk
parity in portfolio construction" (Choi, Kim & Kim, 2021, arXiv 2106.09055).

Self-contained, PUBLIC data only (Yahoo Finance — the paper's own source).
Reproduces claimed → measured for the headline (Calmar raw-ρ vs equal weight)
plus the punchline the paper never tested: the t-statistic of the edge.

Published verdict: OVERSTATED — the mechanics replicate (10.98% vs claimed
10.32%), but the claimed outperformance over equal weight is statistical noise
(t = 0.69 over 22 years) and the tilt that exists is a momentum loading.
Full report: https://market-graphs.com/praxis/papers/359491
"""
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

# ---------------------------------------------------------------- universe ---
# The paper's Table 1 (point-in-time DJIA membership), verbatim — including its
# holes: GM (member 1999-2009) has no Yahoo data before Nov-2010 (new-GM), KRFT
# none at all — true in the paper's own 2021 download too. UTX is served via the
# RTX price lineage (2020 rename). WBA went private in 2025 and its public data
# disappeared (1 name for 2.5 of 22 years; <5bp/yr impact).
MEMBERSHIP = [
    ("AA", "1999-01-01", "2013-09-23"), ("AAPL", "2015-03-19", "2020-12-31"),
    ("AIG", "2004-04-08", "2008-09-22"), ("AMGN", "2020-08-31", "2020-12-31"),
    ("AXP", "1999-01-01", "2020-12-31"), ("BA", "1999-01-01", "2020-12-31"),
    ("BAC", "2008-02-19", "2013-09-23"), ("C", "1999-01-01", "2009-06-08"),
    ("CAT", "1999-01-01", "2020-12-31"), ("CRM", "2020-08-31", "2020-12-31"),
    ("CSCO", "2009-06-08", "2020-12-31"), ("DD", "1999-01-01", "2019-04-02"),
    ("DIS", "1999-01-01", "2020-12-31"), ("DOW", "2019-04-02", "2020-12-31"),
    ("GE", "1999-01-01", "2018-06-26"), ("GM", "1999-01-01", "2009-06-08"),
    ("GS", "2013-09-23", "2020-12-31"), ("GT", "1999-01-01", "1999-11-01"),
    ("HD", "1999-11-01", "2020-12-31"), ("HON", "1999-01-01", "2008-02-19"),
    ("HON", "2020-08-31", "2020-12-31"), ("HPQ", "1999-01-01", "2013-09-23"),
    ("IBM", "1999-01-01", "2020-12-31"), ("INTC", "1999-11-01", "2020-12-31"),
    ("IP", "1999-01-01", "2004-04-08"), ("JNJ", "1999-01-01", "2020-12-31"),
    ("JPM", "1999-01-01", "2020-12-31"), ("KO", "1999-01-01", "2020-12-31"),
    ("KRFT", "2008-09-22", "2012-09-24"), ("MCD", "1999-01-01", "2020-12-31"),
    ("MMM", "1999-01-01", "2020-12-31"), ("MO", "1999-01-01", "2008-02-19"),
    ("MRK", "1999-01-01", "2020-12-31"), ("MSFT", "1999-11-01", "2020-12-31"),
    ("NKE", "2013-09-23", "2020-12-31"), ("PFE", "2004-04-08", "2020-08-31"),
    ("PG", "1999-01-01", "2020-12-31"), ("RTX", "2020-04-06", "2020-08-31"),
    ("T", "1999-11-01", "2004-04-08"), ("T", "2005-11-21", "2015-03-19"),
    ("TRV", "2009-06-08", "2020-12-31"), ("UNH", "2012-09-24", "2020-12-31"),
    ("UTX", "1999-01-01", "2020-04-06"), ("V", "2013-09-23", "2020-12-31"),
    ("VZ", "2004-04-08", "2020-12-31"),
    ("WMT", "1999-01-01", "2020-12-31"), ("XOM", "1999-01-01", "2020-08-31"),
]
ALIAS = {"UTX": "RTX"}
START, END, FIRST_FORMATION = "1998-06-01", "2020-12-31", "1999-06-01"

# ------------------------------------------------------------------- data ---
print("Fetching Yahoo adjusted daily prices (the paper's own data source)…")
tickers = sorted({t for t, _, _ in MEMBERSHIP})
raw = yf.download(sorted({ALIAS.get(t, t) for t in tickers} | {"^IRX"}),
                  start=START, end="2021-01-01", auto_adjust=True, progress=False)["Close"]
prices = pd.DataFrame({t: raw[ALIAS.get(t, t)] for t in tickers if ALIAS.get(t, t) in raw})
prices.index = pd.to_datetime(prices.index).tz_localize(None)
rf_daily = (raw["^IRX"] / 100.0 / 252.0).reindex(prices.index).ffill().fillna(0.0)

windows = {}
for t, lo, hi in MEMBERSHIP:
    windows.setdefault(t, []).append((pd.Timestamp(lo), pd.Timestamp(hi)))


In [ ]:
# ------------------------------------------------ scores (6-month Calmar) ---
month_ends = prices.loc[FIRST_FORMATION:END].groupby(
    [prices.loc[FIRST_FORMATION:END].index.year,
     prices.loc[FIRST_FORMATION:END].index.month]).apply(lambda x: x.index[-1]).tolist()

def eligible(t):
    lo = t - pd.DateOffset(months=6)
    win = prices.loc[lo:t]
    return [tk for tk, ws in windows.items()
            if tk in prices.columns and any(a <= t <= b for a, b in ws)
            and win[tk].notna().sum() >= 100]

def calmar_scores(t):
    lo = t - pd.DateOffset(months=6)
    win = prices.loc[lo:t, eligible(t)].ffill()
    cum = win.iloc[-1] / win.iloc[0] - 1.0
    wealth = win / win.iloc[0]
    mdd = -(wealth / wealth.cummax() - 1.0).min()
    return cum / mdd.replace(0.0, np.nan)

def weights(scores, rule):
    s = scores.dropna()
    w = pd.Series(1.0, index=s.index) if rule == "ew" else s.clip(lower=0.0)
    tot = float(w.sum())
    if not np.isfinite(tot) or tot <= 0:
        w = pd.Series(1.0, index=s.index); tot = float(w.sum())
    return w / tot

# --------------------------------- 6/6 monthly overlapping-tranche engine ---
def tranche_backtest(wbf, n_sleeves=6):
    """Six sleeves, one re-formed per month, buy-and-hold drift within a sleeve.
    Weights formed at close(t) earn from t+1 — no look-ahead."""
    fdates = sorted(wbf)
    cols = prices.columns
    days = prices.index
    R1 = 1.0 + prices.pct_change().fillna(0.0).values
    pos = {d: i for i, d in enumerate(days)}
    fpos = [pos[d] for d in fdates]
    D = len(days)
    port = np.zeros(D)
    sleeve_turns = []

    for k in range(n_sleeves):
        events = [0] + [i for i in range(1, len(fdates)) if i % n_sleeves == k]
        v = 1.0 / n_sleeves
        for e, i in enumerate(events):
            w = wbf[fdates[i]].reindex(cols).fillna(0.0).values
            a = fpos[i]
            b = fpos[events[e + 1]] if e + 1 < len(events) else D - 1
            if b <= a:
                continue
            C = np.cumprod(R1[a + 1: b + 1], axis=0)
            path = C @ w
            port[a + 1: b + 1] += v * path
            v = v * path[-1]
            if e + 1 < len(events):
                drift = w * C[-1]; drift = drift / max(drift.sum(), 1e-12)
                w_new = wbf[fdates[events[e + 1]]].reindex(cols).fillna(0.0).values
                sleeve_turns.append(0.5 * np.abs(w_new - drift).sum())
    wealth = pd.Series(port[fpos[0] + 1:], index=days[fpos[0] + 1:])
    r = wealth.pct_change(); r.iloc[0] = wealth.iloc[0] - 1.0
    return r, float(np.mean(sleeve_turns)) * 100

print("Scoring 259 formations + running the 6-sleeve overlapping backtest…")
scores = {t: calmar_scores(t) for t in month_ends}
r_f, turn_f = tranche_backtest({t: weights(s, "raw") for t, s in scores.items()})
r_e, turn_e = tranche_backtest({t: weights(s, "ew") for t, s in scores.items()})

# ---------------------------------------------------------------- results ---
def stats(r):
    ret = float(r.mean() * 252 * 100)
    vol = float(r.std(ddof=1) * np.sqrt(252) * 100)
    wealth = (1 + r).cumprod()
    mdd = float((wealth / wealth.cummax() - 1).min() * 100)
    return ret, vol, ret / vol, mdd

f, e = stats(r_f), stats(r_e)
am = ((1 + r_f).resample("ME").prod() - (1 + r_e).resample("ME").prod()).dropna()
t_spread = float(am.mean() / am.std(ddof=1) * np.sqrt(len(am)))

print(f"""
================= claimed → measured (DJIA, 1999–2020) =================
                          paper       this notebook
Calmar raw-ρ return       10.32%/yr   {f[0]:.2f}%/yr
Equal-weight benchmark     9.84%/yr   {e[0]:.2f}%/yr
Calmar vol / EW vol       18.68/19.82 {f[1]:.1f}/{e[1]:.1f}
Calmar MDD / EW MDD       −39.5/−57.8 {f[3]:.1f}/{e[3]:.1f}
'Sharpe' (ret/vol, as tabled) 0.552/0.497 {f[2]:.3f}/{e[2]:.3f}
Turnover (sleeve basis)   66.1%/6.75% {turn_f:.1f}%/{turn_e:.1f}%
Active spread             +48bp       {(f[0]-e[0])*100:+.0f}bp gross

The number the paper never reports:
t-stat of the monthly active spread over 22 years: {t_spread:.2f}
(a real edge needs ~2; the multiple-testing bar across their 46-config grid is 3)
=========================================================================""")


## What you just reproduced
- **Calmar raw-ρ: 10.98%/yr vs equal weight 9.96%** (paper: 10.32 vs 9.84) — the headline *reproduces*; the gap to the paper's numbers is 2026-vs-2021 Yahoo dividend-adjustment drift, which itself moves a sub-50bp effect around.
- **Max drawdown −38.5% vs −57.6%** — the drawdown improvement is real; the Calmar tilt does shallow the hole.
- **The t-stat of the active spread: 0.69 over 22 years.** The paper never tests its own core claim ("outperforms the equally-weighted portfolio"). At a ~6%/yr tracking error — annual actives swing from −1441bp (2000) to +1845bp (2008) — a +100bp average is noise. Our full battery adds: Deflated Sharpe 0.43 (needs 0.95), PBO 0.61, and momentum spanning (the tilt is β(Mom) = 0.18, t = 4.0, with zero residual alpha).

**Verdict: OVERSTATED — honest mechanics, insignificant claim.** The full C0–C7 ladder, the 46-config grid, the universe forensics (the point-in-time table's holes), and the post-publication 2021–2026 frontier run are in the report: https://market-graphs.com/praxis/papers/359491
